## SAM 3 for Video

Segment Anything Model 3 (SAM 3), the shift from geometric promptable segmentation to open-vocabulary concept segmentation, and why that matters.

In [22]:
import cv2
import torch
import numpy as np
import gradio as gr

from accelerate import Accelerator
from transformers import Sam3TrackerVideoModel, Sam3TrackerVideoProcessor

## ── Model setup


In [23]:
device = Accelerator().device
model = Sam3TrackerVideoModel.from_pretrained(
    "facebook/sam3"
).to(device, dtype=torch.bfloat16)
processor = Sam3TrackerVideoProcessor.from_pretrained("facebook/sam3")

Loading weights:   0%|          | 0/845 [00:00<?, ?it/s]

## ── Video loader (OpenCV) 

In [24]:
def load_video_opencv(video_path, max_frames=300):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if max_frames and len(frames) >= max_frames:
            break

    cap.release()

    if len(frames) == 0:
        raise ValueError("Video opened but 0 frames decoded — check codec")

    return frames, fps

## ── Extract first frame

In [25]:
def extract_first_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return None

## ── Tracking function 

In [26]:
def track_object(video_path, point_coords):
    if video_path is None:
        return None, "❌ Please upload a video first"

    if (
        point_coords is None
        or not hasattr(point_coords, "__len__")
        or len(point_coords) < 2
    ):
        return None, "❌ Please click on the first frame to select a point"

    try:
        x, y = int(point_coords[0]), int(point_coords[1])
    except (TypeError, IndexError, ValueError):
        return None, "❌ Invalid point — please click the frame again"

    try:
        video_frames, fps = load_video_opencv(video_path, max_frames=300)
        print(f"[INFO] Loaded {len(video_frames)} frames at {fps:.1f} FPS")

        h, w = video_frames[0].shape[:2]

        # Init SAM3 session
        inference_session = processor.init_video_session(
            video=video_frames,
            inference_device=device,
            dtype=torch.bfloat16,
        )

        processor.add_inputs_to_inference_session(
            inference_session=inference_session,
            frame_idx=0,
            obj_ids=1,
            input_points=[[[[x, y]]]],
            input_labels=[[[1]]],
        )

        # Segment frame 0
        outputs     = model(inference_session=inference_session, frame_idx=0)
        first_masks = processor.post_process_masks(
            [outputs.pred_masks],
            original_sizes=[[
                inference_session.video_height,
                inference_session.video_width,
            ]],
            binarize=False,
        )[0]

        video_segments = {0: first_masks}

        # Propagate through all frames
        for out in model.propagate_in_video_iterator(inference_session):
            masks = processor.post_process_masks(
                [out.pred_masks],
                original_sizes=[[
                    inference_session.video_height,
                    inference_session.video_width,
                ]],
                binarize=False,
            )[0]
            video_segments[out.frame_idx] = masks

        # Write annotated output video
        output_path = "output_tracked.mp4"
        writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (w, h),
        )

        for idx, frame in enumerate(video_frames):
            frame = frame.copy().astype(np.uint8)
            if idx in video_segments:
                mask          = video_segments[idx][0, 0].float().cpu().numpy() > 0.0
                overlay       = frame.copy()
                overlay[mask] = [255, 0, 0]
                frame = cv2.addWeighted(
                    frame.astype(np.float32),   0.6,
                    overlay.astype(np.float32), 0.4,
                    0,
                ).astype(np.uint8)
            writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

        writer.release()
        print(f"[INFO] Output saved to {output_path}")
        return (
            output_path,
            f"✅ Tracked {len(video_segments)} frames at point ({x}, {y})",
        )

    except Exception as e:
        import traceback
        return None, f"❌ {str(e)}\n{traceback.format_exc()}"

## ── Event handlers

In [27]:
def on_video_upload(video):
    if video:
        frame = extract_first_frame(video)
        if frame is None:
            return None, None, "❌ Could not read first frame — check video format"
        return frame, None, "✅ Loaded. Now click the object you want to track."
    return None, None, ""

def on_click(img, evt: gr.SelectData):
    x, y     = evt.index[0], evt.index[1]
    img_copy = img.copy()
    cv2.circle(img_copy, (x, y),  8, (255,   0,   0), -1)
    cv2.circle(img_copy, (x, y), 11, (255, 255, 255),  2)
    return img_copy, (x, y), f"📍 Point selected: ({x}, {y})"

## ── Gradio UI 

In [28]:
with gr.Blocks(title="SAM3 Video Tracker") as demo:

    gr.Markdown("# 🎯 SAM3 Video Object Tracker")
    gr.Markdown(
        "**Step 1** Upload video → "
        "**Step 2** Click object on first frame → "
        "**Step 3** Hit Track"
    )

    clicked_point = gr.State(None)

    with gr.Row():
        with gr.Column():
            video_input   = gr.Video(label="Upload Video")
            first_frame   = gr.Image(
                label="Click on object to track", type="numpy"
            )
            point_display = gr.Textbox(
                label="Selected Point", interactive=False
            )
            track_btn     = gr.Button("Track Object", variant="primary")

        with gr.Column():
            video_output  = gr.Video(label="Tracked Video")
            status_output = gr.Textbox(label="Status", lines=6)

    video_input.change(
        fn=on_video_upload,
        inputs=[video_input],
        outputs=[first_frame, clicked_point, status_output],
    )

    first_frame.select(
        fn=on_click,
        inputs=[first_frame],
        outputs=[first_frame, clicked_point, point_display],
    )

    track_btn.click(
        fn=track_object,
        inputs=[video_input, clicked_point],
        outputs=[video_output, status_output],
    )

## ── Entry point

In [29]:
if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        debug=True,
        share=False,
    )

ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7860): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.